# Evaluating Agent Behavior Systematically

Does the agent work? This is the most important question in agent development, and the hardest to answer. Vibes-based testing — running the agent on a few examples and seeing if it *looks* right — doesn't scale. It misses regressions, obscures cost, and provides no signal about the agent's behavior on the long tail of inputs you haven't tried.

Systematic evaluation requires four things: (1) a **task suite** with known expected outcomes, (2) a **trajectory recorder** that captures the full sequence of events — not just the final answer, (3) a **cost tracker** that treats token spend as a first-class metric alongside quality, and (4) an **LLM judge** for tasks where there is no single correct answer and programmatic verification falls short.

This notebook builds a mini evaluation framework from scratch using the CDA library. We define a `Task` dataclass and `TaskResult` record, implement a `run_task()` harness that captures complete trajectories, write an LLM-as-judge to score open-ended responses, extract trajectory metrics that reveal *how* the agent worked, and track cost per task to enable model comparisons.

## Why Agent Evaluation Is Hard

Standard ML evaluation — split a dataset, compute accuracy, report a number — does not transfer cleanly to agents. Five properties of the agent setting make evaluation qualitatively harder:

1. **Non-determinism.** The same task, the same agent, the same input yields different results across runs when temperature $> 0$. A single-run pass/fail verdict is unreliable. You need multiple runs to estimate accuracy as a distribution, not a point.

2. **Multi-step trajectories.** The final answer is only part of the story. An agent that answers correctly after 10 tool calls — three of which were failed retries — is less reliable than one that answers correctly in 2. Binary pass/fail is blind to this.

3. **Partial credit.** An agent that reads the right files, identifies the root cause, but writes a subtly wrong patch deserves more credit than one that gives up immediately. Hard pass/fail loses this signal, which is essential for diagnosing and improving specific failure modes.

4. **No ground truth.** For open-ended tasks — "refactor this module", "summarize this document", "write a test suite" — there is no canonical correct answer. Programmatic checks are either impossible or would require as much effort as the task itself.

5. **Cost versus quality.** A model that achieves 95% pass rate at $\$0.10$ per task may be strictly dominated by one that achieves 90% at $\$0.01$ per task, depending on the application. Ignoring cost gives an incomplete picture.

:::{.callout-important}
The remedy is a task suite with diverse tasks, clear verification functions where possible, an LLM judge for ambiguous cases, and cost tracked alongside quality on every run. Evaluation is not optional — it is the only way to know whether a change is an improvement.

:::

## Defining a Task Suite

**Setup.** Imports and client initialization:

In [ ]:
import os
import json
import time
import asyncio
from dataclasses import dataclass, field
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)
MODEL = "anthropic/claude-sonnet-4"

CDA imports:

In [ ]:
from notebooks.agent.config import Config, ApprovalPolicy
from notebooks.agent.agent import Agent
from notebooks.agent.events import AgentEvent, AgentEventType, TokenUsage

A `Task` captures everything needed to run and verify a single evaluation unit. A `TaskResult` captures everything produced by running it — events, timing, usage, and the verdict:

In [ ]:
@dataclass
class Task:
    name: str
    description: str
    user_message: str
    expected_output: str | None = None          # for display / reference
    category: str = "general"                  # factual | manipulation | reasoning | code
    difficulty: str = "medium"                 # easy | medium | hard
    verify: callable = field(default=None)     # optional programmatic check


@dataclass
class TaskResult:
    task: Task
    events: list[AgentEvent]
    final_response: str
    passed: bool | None                        # None if no verify function
    time_seconds: float
    usage: TokenUsage
    error: str | None = None

**Task suite.** We define five tasks spanning three difficulty levels and four categories. Each task that has an unambiguous correct answer gets a `verify` lambda for programmatic checking:

In [ ]:
def build_task_suite() -> list[Task]:
    tasks = []

    # --- Easy: 1 tool call ---
    tasks.append(Task(
        name="arithmetic",
        description="Compute 37 × 43 using the calculator tool",
        user_message="What is 37 * 43? Use a tool to compute it.",
        expected_output="1591",
        category="factual",
        difficulty="easy",
        verify=lambda r: "1591" in r,
    ))

    tasks.append(Task(
        name="list_directory",
        description="List files in the current working directory",
        user_message="List the files and directories in the current working directory.",
        category="manipulation",
        difficulty="easy",
        verify=lambda r: len(r) > 20,  # any non-trivial response passes
    ))

    # --- Medium: 2-3 tool calls ---
    tasks.append(Task(
        name="write_and_read",
        description="Write a file and confirm its contents",
        user_message="Create a file called /tmp/eval_test.txt containing the text 'Hello, evaluation!' and then read it back to confirm the contents.",
        expected_output="Hello, evaluation!",
        category="manipulation",
        difficulty="medium",
        verify=lambda r: "Hello, evaluation!" in r,
    ))

    tasks.append(Task(
        name="reasoning_chain",
        description="Multi-step arithmetic requiring chained tool calls",
        user_message="Compute (15 + 27) * (8 - 3). Show each step using the calculator tool.",
        expected_output="210",
        category="reasoning",
        difficulty="medium",
        verify=lambda r: "210" in r,
    ))

    # --- Hard: requires planning ---
    tasks.append(Task(
        name="file_analysis",
        description="Analyze project structure and count Python files",
        user_message="How many Python (.py) files are in the src/ directory? Count them using the glob or shell tool.",
        category="reasoning",
        difficulty="hard",
        verify=lambda r: any(c.isdigit() for c in r),  # should mention a number
    ))

    return tasks


TASK_SUITE = build_task_suite()
print(f"Task suite: {len(TASK_SUITE)} tasks")
for t in TASK_SUITE:
    print(f"  [{t.difficulty:6s}] {t.name} ({t.category})")

## Running & Recording

We implement the `run_task()` harness. It drives the CDA `Agent` on a single task, collects every `AgentEvent` into a list, extracts the final response and token usage from the relevant events, and runs the `verify` lambda to produce a pass/fail verdict:

In [ ]:
async def run_task(agent: Agent, task: Task) -> TaskResult:
    """Run a single task and record the full trajectory."""
    events = []
    final_response = ""
    usage = TokenUsage()
    error = None
    start = time.perf_counter()

    try:
        async for event in agent.run(task.user_message):  # <1>
            events.append(event)
            if event.type == AgentEventType.TEXT_COMPLETE:
                final_response = event.data.get("content", "")
            elif event.type == AgentEventType.AGENT_END:
                if event.data.get("usage"):
                    u = event.data["usage"]
                    usage = TokenUsage(
                        prompt_tokens=u.get("prompt_tokens", 0),
                        completion_tokens=u.get("completion_tokens", 0),
                        total_tokens=u.get("total_tokens", 0),
                        cached_tokens=u.get("cached_tokens", 0),
                    )
            elif event.type == AgentEventType.AGENT_ERROR:
                error = event.data.get("error", "unknown error")
    except Exception as e:
        error = str(e)

    elapsed = time.perf_counter() - start
    passed = task.verify(final_response) if task.verify and final_response else None

    return TaskResult(
        task=task,
        events=events,
        final_response=final_response,
        passed=passed,
        time_seconds=elapsed,
        usage=usage,
        error=error,
    )

1. `agent.run(task.user_message)` — the CDA `Agent`'s async generator interface. We iterate through all emitted events, accumulating them in a list so trajectory analysis can replay the run later.

We instantiate a single agent with `YOLO` approval and run it across all five tasks. Between tasks, we reset the session so each task starts with a clean conversation:

In [ ]:
config = Config(approval=ApprovalPolicy.YOLO)  # <1>
agent = Agent(config)

results = []
for task in TASK_SUITE:
    print(f"Running: {task.name}...", end=" ", flush=True)
    agent.session.reset()  # <2>
    result = await run_task(agent, task)
    results.append(result)
    status = "✓" if result.passed else ("✗" if result.passed is False else "?")
    print(f"{status} ({result.time_seconds:.1f}s, {result.usage.total_tokens} tokens)")

1. `YOLO` policy so the agent executes all tool calls without pausing for human confirmation.
2. Reset the session between tasks — each task starts with a fresh conversation history.

Summarizing the results as a table:

In [ ]:
print(f"{'Task':20s} {'Cat':12s} {'Diff':6s} {'Pass':6s} {'Turns':6s} {'Tokens':8s} {'Time':6s}")
print("-" * 70)
for r in results:
    turns = sum(1 for e in r.events if e.type == AgentEventType.TOOL_CALL_START)
    pass_str = "✓" if r.passed else ("✗" if r.passed is False else "?")
    print(f"{r.task.name:20s} {r.task.category:12s} {r.task.difficulty:6s} {pass_str:6s} {turns:6d} {r.usage.total_tokens:8d} {r.time_seconds:6.1f}s")

passed = sum(1 for r in results if r.passed)
total = sum(1 for r in results if r.passed is not None)
print(f"\nPass rate: {passed}/{total} ({100 * passed // total if total else 0}%)")

## LLM-as-Judge

Programmatic verification works well for tasks with a single correct answer — arithmetic results, expected file contents, numeric counts. For open-ended responses, we need a second LLM to evaluate quality. We call this the **judge**.

The judge receives the original task description and the agent's response, then assigns a score from 1 to 5 using a fixed rubric:

- **1** — completely fails to address the task
- **2** — partially addresses the task with major gaps or errors
- **3** — addresses the task but with notable mistakes or incompleteness
- **4** — addresses the task correctly with only minor issues
- **5** — perfect: exactly what was asked for, nothing missing

We request a temperature of $0$ and constrain `max_tokens` to $5$ so the model returns only the integer score:

In [ ]:
JUDGE_SYSTEM = """You are an expert evaluator of AI agent responses. Score the agent's response on a scale of 1-5.

Rubric:
1 = Completely fails to address the task
2 = Partially addresses the task with major gaps or errors
3 = Addresses the task but with notable mistakes or incompleteness
4 = Addresses the task correctly with only minor issues
5 = Perfect — exactly what was asked for, nothing missing

Return only a single integer (1-5). No explanation."""


async def judge(task: Task, response: str) -> int:
    prompt = f"Task: {task.user_message}\n\nAgent response:\n{response}"
    completion = await client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        max_tokens=5,
        temperature=0.0,
    )
    raw = completion.choices[0].message.content.strip()
    try:
        score = int(raw[0])  # <1>
        return max(1, min(5, score))
    except (ValueError, IndexError):
        return 3  # <2>

1. Extract the first character in case the model adds punctuation like `"4."` or `"4/5"`.
2. Default to the middle score if parsing fails — a conservative choice that avoids penalizing tasks where the judge response was malformed.

We calibrate the judge against our programmatic results by running it on every task that has a `verify` function. Treating a judge score of $\geq 4$ as passing, we compare the two verdicts:

In [ ]:
print(f"{'Task':20s} {'Programmatic':15s} {'Judge':8s} {'Agree?':8s}")
print("-" * 55)

for r in results:
    if r.passed is not None and r.final_response:
        score = await judge(r.task, r.final_response)
        judge_pass = score >= 4  # <1>
        agree = judge_pass == r.passed
        print(f"{r.task.name:20s} {'✓' if r.passed else '✗':15s} {score}/5{'':3s} {'✓' if agree else '✗'}")

1. Treat score $\geq 4$ as "passing" so we can compare against the binary programmatic verdict on the same scale.

:::{.callout-note}
LLM judges are most valuable for tasks without a single correct answer — code style, explanation quality, research summaries. For exact-match tasks like "compute 37 * 43", programmatic verification is faster, cheaper, and more reliable. The two approaches are complementary: use programmatic checks where possible, and reserve the judge for tasks where no clean check exists.

:::

## Trajectory Analysis

Pass/fail answers whether the agent succeeded. Trajectory analysis answers *how*. Because `run_task()` captured every `AgentEvent`, we can replay each run and extract metrics that reveal efficiency, error recovery, and tool diversity — all without re-running the agent.

We define `TrajectoryMetrics` and `analyze_trajectory()` to compute these from a stored `TaskResult`:

In [ ]:
@dataclass
class TrajectoryMetrics:
    task_name: str
    turn_count: int           # number of LLM response turns (tool-call turns + final turn)
    tool_calls: int           # total tool invocations
    error_tool_calls: int     # tool calls that returned errors
    unique_tools: set[str]    # set of distinct tool names used
    recovered: bool           # True if there were errors but the task ultimately passed


def analyze_trajectory(result: TaskResult) -> TrajectoryMetrics:
    """Extract trajectory metrics from a recorded TaskResult."""
    tool_starts = [e for e in result.events if e.type == AgentEventType.TOOL_CALL_START]
    tool_completes = [e for e in result.events if e.type == AgentEventType.TOOL_CALL_COMPLETE]

    turn_count = sum(1 for e in result.events if e.type == AgentEventType.TEXT_COMPLETE)
    tool_calls = len(tool_starts)
    error_calls = sum(1 for e in tool_completes if not e.data.get("success", True))
    unique = {e.data.get("name", "") for e in tool_starts}
    recovered = error_calls > 0 and (result.passed is True)

    return TrajectoryMetrics(
        task_name=result.task.name,
        turn_count=turn_count,
        tool_calls=tool_calls,
        error_tool_calls=error_calls,
        unique_tools=unique,
        recovered=recovered,
    )


# Analyze all results
metrics = [analyze_trajectory(r) for r in results]
print(f"{'Task':20s} {'Turns':6s} {'Tools':6s} {'Errors':7s} {'Tool names':30s}")
print("-" * 75)
for m in metrics:
    tools_str = ", ".join(sorted(m.unique_tools)) if m.unique_tools else "(none)"
    print(f"{m.task_name:20s} {m.turn_count:6d} {m.tool_calls:6d} {m.error_tool_calls:7d} {tools_str[:28]:30s}")

:::{.callout-note}
Trajectory analysis catches problems that pass/fail misses. An agent that gets the right answer after 8 failed tool calls is less reliable than one that gets it in 2 — it has higher latency, higher cost, and a larger attack surface for prompt injection through tool error messages. High `error_tool_calls` rates often indicate schema mismatches or unclear tool descriptions.

:::

## Cost Tracking

Token cost must be a first-class metric. A model that achieves perfect accuracy but costs $10\times$ more per task may not be the right choice for production workloads that run thousands of tasks per day. We estimate cost from the `TokenUsage` records captured during the run — no additional LLM calls are needed.

Approximate pricing for common models is given below in USD per $10^6$ tokens:

In [ ]:
# Approximate pricing (USD per 1M tokens)
PRICING = {
    "anthropic/claude-sonnet-4":  {"prompt": 3.0,   "completion": 15.0},
    "anthropic/claude-haiku-3-5": {"prompt": 0.8,   "completion": 4.0},
    "openai/gpt-4o":              {"prompt": 2.5,   "completion": 10.0},
    "openai/gpt-4o-mini":         {"prompt": 0.15,  "completion": 0.60},
    "google/gemini-flash-1-5":    {"prompt": 0.075, "completion": 0.30},
}


def cost_estimate(usage: TokenUsage, model: str) -> float:
    """Estimate cost in USD from token usage and model name."""
    prices = PRICING.get(model, {"prompt": 3.0, "completion": 15.0})  # <1>
    prompt_cost = usage.prompt_tokens * prices["prompt"] / 1_000_000
    completion_cost = usage.completion_tokens * prices["completion"] / 1_000_000
    return prompt_cost + completion_cost


# Compute costs for our results
total_cost = 0.0
print(f"{'Task':20s} {'Prompt':8s} {'Compl':8s} {'Total':8s} {'Cost ($)':10s}")
print("-" * 60)
for r in results:
    cost = cost_estimate(r.usage, MODEL)
    total_cost += cost
    print(f"{r.task.name:20s} {r.usage.prompt_tokens:8d} {r.usage.completion_tokens:8d} {r.usage.total_tokens:8d} ${cost:.5f}")
print(f"\n{'Total suite cost':48s} ${total_cost:.5f}")

1. Default to claude-sonnet-4 pricing when the model is not in the table — a conservative fallback that overestimates rather than underestimates.

:::{.callout-note}
On a 5-task suite, costs are negligible. But the same agent running on 1,000 tasks per day accumulates quickly. A model that costs $10\times$ less and passes 90% of tasks versus 95% may be the right choice, depending on the cost of a failed task in the target application. Always measure cost alongside quality — the two together determine value.

:::

---

■